# Run 2 — Dimension Updates + Significant Fact Append (`batch_id = 2`)

Demonstrates incremental processing:
- **Dimension updates (effective 1996-01-01):** a few customers (market segment + address) and
  suppliers (name) change, producing new SCD2 versions in silver and gold. The change carries a
  **business effective date** (`effective_date`), so gold facts attribute orders placed on/after
  1996 to the new version and earlier orders to the baseline version (point-in-time).
- **Fact append:** a large batch of brand-new orders + line items lands and flows through to
  `fct_order_lines`, growing the star schema.
- **Schema evolution:** the `customer` batch introduces a new upstream column (`loyalty_tier`)
  that did not exist on day 1. Auto Loader (`schemaEvolutionMode: addNewColumns`) adds it to
  `bronze_crm.customer` automatically — no spec change.
- **Late-arriving dimension:** one line item references a part (`9000001`) whose master record
  arrives only in Run 3, exercising the unknown-member (`-1`) handling in `dim_part` /
  `fct_order_lines`.
- **Data-quality quarantine:** a handful of deliberately malformed `customer_address` and `orders`
  rows are mixed into the batch. The silver expectations (`quarantineMode: table`) drop them from
  the clean tables and route them to `silver.customer_address_quarantine` /
  `silver.orders_quarantine` for inspection.

In [ ]:
%run ./initialize

In [ ]:
# Focused dimension updates (a handful of keys) -> new SCD2 versions
batch_id = 2
load_ts = datetime.now()

# Business effective date for the Run 2 changes (on the 1992-1998 order timeline). Orders placed
# on/after this date resolve to these new versions; earlier orders keep the baseline version.
RUN2_EFFECTIVE = "1996-01-01 00:00:00"

demo_customer_keys = "(1, 2, 3)"
demo_supplier_keys = "(1, 2)"

# Customer: change market segment (effective 1996).
# Schema evolution: this batch ALSO introduces a brand-new upstream column, `loyalty_tier`,
# that did not exist in the day-1 load. Bronze reads Parquet with
# cloudFiles.schemaEvolutionMode=addNewColumns, so bronze_crm.customer gains the column
# automatically (earlier rows read back NULL) with no spec change. Silver keeps its explicit
# contract and does not project the new column -- a deliberate schema-on-read-bronze vs
# contract-bound-silver contrast.
cust = spark.sql(f"SELECT c_custkey AS customer_id, c_name AS name, c_acctbal AS acctbal, 'AUTOMOBILE' AS mktseg, 'GOLD' AS loyalty_tier FROM {sample_source_schema}.customer WHERE c_custkey IN {demo_customer_keys}")
cust = with_op(with_effective(with_metadata(cust, "crm", batch_id, load_ts), RUN2_EFFECTIVE))
write_landing(cust, "crm", "customer", INCREMENTAL_ZONE, load_ts)

# Customer address: change address (legit demo updates, effective 1996)
addr = spark.sql(f"SELECT c_custkey AS customer_id, concat('UPDATED - ', c_address) AS address, c_nationkey AS nat_id FROM {sample_source_schema}.customer WHERE c_custkey IN {demo_customer_keys}")

# DQ demo: inject a malformed customer_address row that violates a silver expectation
# (nation_key NOT NULL -- a referential-integrity break on an otherwise valid customer_key).
# With quarantineMode=table on silver.customer_address it is dropped from the clean dimension
# and captured in silver.customer_address_quarantine.
# (PK-null is demonstrated on the append-only orders fact instead: a NULL SCD2 key is an
#  apply_changes edge case, so the dimension demo uses an FK violation on a valid key.)
bad_addr = spark.sql("""
    SELECT CAST(900000002 AS BIGINT) AS customer_id, 'DQ_DEMO: null nation_key (FK)' AS address, CAST(NULL AS BIGINT) AS nat_id
""")
addr = addr.unionByName(bad_addr)
addr = with_op(with_effective(with_metadata(addr, "crm", batch_id, load_ts), RUN2_EFFECTIVE))
write_landing(addr, "crm", "customer_address", INCREMENTAL_ZONE, load_ts)

# Supplier: rename (effective 1996)
supp = spark.sql(f"SELECT s_suppkey AS supplier_id, concat(s_name, ' (RENAMED)') AS name, s_acctbal AS acctbal, s_comment AS comment FROM {sample_source_schema}.supplier WHERE s_suppkey IN {demo_supplier_keys}")
supp = with_op(with_effective(with_metadata(supp, "procurement", batch_id, load_ts), RUN2_EFFECTIVE))
write_landing(supp, "procurement", "supplier", INCREMENTAL_ZONE, load_ts)

print("Run 2 dimension updates staged.")

In [ ]:
# Significant fact append: a batch of brand-new orders + their line items.
# New keys are produced by offsetting a deterministic slice of source order keys, so orders
# and line items stay referentially consistent while never colliding with the initial load.
ORDER_OFFSET = 100000000
ORDER_FILTER = "o_orderkey <= 60000"
LINE_FILTER = "l_orderkey <= 60000"

new_orders = spark.sql(f"""
    SELECT o_orderkey + {ORDER_OFFSET} AS o_orderkey, o_custkey, o_orderstatus, o_totalprice,
           date_add(o_orderdate, 2000) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment
    FROM {sample_source_schema}.orders
    WHERE {ORDER_FILTER}
""")
# DQ demo: inject malformed orders that violate the silver expectations
# (order_key NOT NULL, total_price >= 0). With quarantineMode=table on silver.orders,
# these are dropped from the clean fact feed and captured in silver.orders_quarantine.
bad_orders = spark.sql(f"""
    SELECT CAST(NULL AS BIGINT) AS o_orderkey, o_custkey, o_orderstatus, o_totalprice,
           date_add(o_orderdate, 2000) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority,
           'DQ_DEMO: null order_key (PK)' AS o_comment
    FROM {sample_source_schema}.orders WHERE o_orderkey = 1
    UNION ALL
    SELECT o_orderkey + 900000000 AS o_orderkey, o_custkey, o_orderstatus,
           CAST(-99999.99 AS DECIMAL(18,2)) AS o_totalprice,
           date_add(o_orderdate, 2000) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority,
           'DQ_DEMO: negative total_price' AS o_comment
    FROM {sample_source_schema}.orders WHERE o_orderkey = 2
""")
new_orders = new_orders.unionByName(bad_orders)
write_landing(with_metadata(new_orders, "order_mgmt", batch_id, load_ts), "order_mgmt", "orders", INCREMENTAL_ZONE, load_ts)

new_lineitems = spark.sql(f"""
    SELECT l_orderkey + {ORDER_OFFSET} AS l_orderkey, l_partkey, l_suppkey, l_linenumber,
           l_quantity, l_extendedprice, l_discount, l_tax, l_returnflag, l_linestatus,
           date_add(l_shipdate, 2000) AS l_shipdate, date_add(l_commitdate, 2000) AS l_commitdate,
           date_add(l_receiptdate, 2000) AS l_receiptdate, l_shipinstruct, l_shipmode, l_comment
    FROM {sample_source_schema}.lineitem
    WHERE {LINE_FILTER}
""")

# Late-arriving dimension demo: this line item references a part (9000001) whose master record
# has NOT arrived yet -- it lands in Run 3. With an unknown-member (-1) row in dim_part and a
# COALESCE in fct_order_lines, the fact resolves to the Unknown part now; Run-3 facts for 9000001
# resolve to the real part once it arrives. Append-only facts are not retro-repointed, so this
# Run-2 row keeps the unknown member -- realistic streaming-warehouse behaviour. line_number 99
# avoids colliding with the order's real lines.
late_part_line = spark.sql(f"""
    SELECT l_orderkey + {ORDER_OFFSET} AS l_orderkey, CAST(9000001 AS BIGINT) AS l_partkey, l_suppkey,
           CAST(99 AS INT) AS l_linenumber, l_quantity, l_extendedprice, l_discount, l_tax,
           l_returnflag, l_linestatus, date_add(l_shipdate, 2000) AS l_shipdate,
           date_add(l_commitdate, 2000) AS l_commitdate, date_add(l_receiptdate, 2000) AS l_receiptdate,
           l_shipinstruct, l_shipmode,
           'LATE_ARRIVAL_DEMO: part 9000001 not yet in product_catalog' AS l_comment
    FROM {sample_source_schema}.lineitem WHERE l_orderkey = 1 AND l_linenumber = 1
""")
new_lineitems = new_lineitems.unionByName(late_part_line)
write_landing(with_metadata(new_lineitems, "order_fulfillment", batch_id, load_ts), "order_fulfillment", "lineitem", INCREMENTAL_ZONE, load_ts)

print("Run 2 fact append staged (batch_id = 2)")